In [ ]:
import sys

In [ ]:
sys.path.append("/home/gputnam/Diffusion-Anomaly-Detection/diffusion-anomaly")

In [ ]:
from guided_diffusion.script_util import (
    model_and_diffusion_defaults,
    diffusion_defaults,
    create_model_and_diffusion,
    args_to_dict,
    add_dict_to_argparser,
    create_gaussian_diffusion
)

from guided_diffusion.resample import UniformSampler
from guided_diffusion import dist_util

from guided_diffusion.fp16_util import *

from guided_diffusion.respace import space_timesteps

import numpy as np
import matplotlib.pyplot as plt
import torch as th

import h5py

import os

import pickle

In [ ]:
plt.rcParams.update({'font.size': 14})

In [ ]:
# GOOD DATA IMAGE
fname = "/scratch/7DayLifetime/munjung/anomaly-detection-raw-h5/run_18515_sub_1_evt_29363/results-grayNN-ddim/g4-raw-1.pkl"

with open(fname, "rb") as f:
    d = pickle.load(f)

good_data = d[35]["original"].T

plt.imshow(good_data)


In [ ]:
# BAD DATA IMAGE
fname = "/scratch/7DayLifetime/munjung/anomaly-detection-raw-h5/run_18515_sub_1_evt_29363/results-grayNN-ddim/g4-raw-1.pkl"

with open(fname, "rb") as f:
    d = pickle.load(f)

bad_data = d[29]["original"].T

plt.imshow(bad_data)

In [ ]:
np.argmax(bad_data[0, :])

In [ ]:
# GOOD MC IMAGE
path = "/scratch/7DayLifetime/munjung/anomaly-detection/68030020_0/g4-raw-0.h5"

def visualize(img):
    return np.clip(img, -1, 1)

with h5py.File(str(path), "r") as f:
    # scale charge by 500
    arrs = [f[ev]["raw"][:] for ev in f.keys()]
allarrs = []
for arr in arrs:
    nrows, ncols = (512, 512)
    plane_boundaries = [0, 1984, 3968, 5638]
    for planeno, (wlo, whi) in enumerate(zip(plane_boundaries[:-1], plane_boundaries[1:])):
        planearr = arr[wlo:whi, :]
        h, w = planearr.shape
        ll = planearr[:(h//nrows)*nrows, :(w//nrows)*nrows].reshape(h//nrows, nrows, -1, ncols).swapaxes(1,2).reshape(-1, nrows, ncols)
        cscale = [200., 100., 200.][planeno]
        allarrs.append(ll/cscale)
        
imgs = visualize(np.expand_dims(np.concatenate(allarrs), axis=1)).astype(np.float32)

In [ ]:
good_mc = np.squeeze(imgs[404])
plt.imshow(good_mc)

In [ ]:
bad_mc = np.squeeze(imgs[410])

bad_mc[:, 244] = bad_data[:, 244]

plt.imshow(bad_mc)

In [ ]:
imgs = np.expand_dims(np.stack([good_mc, bad_mc, good_data, bad_data]), axis=1)
imgs = th.tensor(imgs, device=dist_util.dev())

In [ ]:
th.set_grad_enabled(False)

In [ ]:
args = model_and_diffusion_defaults()
diffusion_args = diffusion_defaults()

In [ ]:
# MODEL
args["image_size"] = 512
args["num_channels"] = 32
args["class_cond"] = False
args["num_res_blocks"] = 2
args["num_heads"] = 8
args["learn_sigma"] = True
args["use_scale_shift_norm"] = False
args["attention_resolutions"] = "16,32"
args["channel_mult"] = "1,2,4,8,8,8"

# DIFFUSION
diffusion_args["diffusion_steps"] = 1000
diffusion_args["noise_schedule"] = "linear"
diffusion_args["rescale_learned_sigmas"] = False
diffusion_args["rescale_timesteps"] = False

diffusion_args.pop("diffusion_steps")
diffusion_args.pop("timestep_respacing")

# TODO: change?
diffusion_args["learn_sigma"] = True

args = args | diffusion_args

In [ ]:
model, diffusion = create_model_and_diffusion(**args)

In [ ]:
MODEL = "/exp/sbnd/data/users/gputnam/training-SBND/iterE/results/brats2update111000.pt"

In [ ]:
model.load_state_dict(
    dist_util.load_state_dict(MODEL, map_location="cpu")
)

model.to(dist_util.dev())
_ = model.eval()

In [ ]:
T = 100

In [ ]:
noisef = diffusion.ddim_sample_loop_progressive(model, imgs.shape, time=T, noise=imgs, 
                                             reverse=True, progress=True)

noise = list(noisef)

In [ ]:
noised = noise[-1]["sample"]

In [ ]:
genf = diffusion.ddim_sample_loop_progressive(model, noised.shape, time=T, noise=noised, progress=True)

gen = list(genf)

In [ ]:
reco = gen[-1]["sample"]

In [ ]:
titles = [
    "Healthy Simulated Image",
    "Diseased Simulated Image",
    "Healthy Data Image",
    "Diseased Data Image",

]

In [ ]:
for i, title in enumerate(titles):
    plt.figure(i)
    fig, axs = plt.subplots(1, 3, figsize=(12,4))
    axs[0].imshow(np.squeeze(imgs[i].cpu().numpy()), vmin=-0.5, vmax=0.5)
    axs[1].imshow(np.squeeze(reco[i].cpu().numpy()), vmin=-0.5, vmax=0.5)
    axs[2].imshow(np.squeeze((imgs[i] - reco[i]).cpu().numpy()), vmin=-0.1, vmax=0.1, cmap="bwr")
    
    axs[1].set_yticks([])
    axs[2].set_yticks([])
    
    axs[0].set_title("Image")
    axs[1].set_title("Reconstructed")
    axs[2].set_title("Sailiency")
    
    axs[0].set_ylabel("Time Tick")
    axs[1].set_xlabel("Wire Number")
    plt.suptitle(title, y=1.02)
    
    fig.subplots_adjust(wspace=0.02, hspace=0.02) # Make room on the right